# Variational Autoencoders

This notebook accompanies the **ML Viz** lesson on VAEs.
We'll implement a VAE with the reparameterization trick and explore
how it structures the latent space.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/generative-models/03-variational-autoencoders

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — an autoencoder with an organized latent space

The plain autoencoder's latent space had holes; the **VAE** fills them by making the encoder output a
**distribution** — a mean `μ` and variance `σ²` per input — and adding a **KL penalty** that pulls
every posterior toward the prior `N(0, I)`. The loss is the **ELBO**: reconstruction error + KL. The
reconstruction term wants informative latents; the KL term wants them packed into a standard Gaussian —
the tension produces a smooth, hole-free latent space you can *sample* from. Training needs one trick:
sampling isn't differentiable, so **reparameterize** `z = μ + σ·ε` with `ε ~ N(0,1)` — randomness moves
to an input, and gradients flow through `μ` and `σ`. We build the whole thing and verify the KL math
by Monte Carlo.

## The Reparameterization Trick

In a VAE, the encoder outputs parameters of a distribution $q(z|x) = \mathcal{N}(\mu, \sigma^2)$.
To sample $z$ while keeping the operation differentiable:

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Let's visualize this.

In [ ]:
np.random.seed(42)

mu, sigma = 2.0, 0.8
n_samples = 500

# Without reparameterization (non-differentiable)
z_wrong = np.random.normal(mu, sigma, n_samples)

# With reparameterization (differentiable)
eps = np.random.normal(0, 1, n_samples)
z_right = mu + sigma * eps

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Reparameterization Trick', color='white', fontsize=13, y=1.02)

# Epsilon distribution
axes[0].hist(eps, bins=30, color='#818cf8', alpha=0.8, edgecolor='#1a1d27')
axes[0].set_title('$\epsilon \sim \mathcal{N}(0, 1)$', color='white', fontsize=12)
axes[0].set_xlabel('$\epsilon$')

# Transformed distribution
axes[1].hist(z_right, bins=30, color='#14b8a6', alpha=0.8, edgecolor='#1a1d27')
x_range = np.linspace(-2, 5, 100)
axes[1].plot(x_range, norm.pdf(x_range, mu, sigma) * n_samples * 0.4, '--', color='white', alpha=0.5)
axes[1].set_title(f'$z = \mu + \sigma \cdot \epsilon$ ($\mu$={mu}, $\sigma$={sigma})', color='white', fontsize=12)
axes[1].set_xlabel('$z$')

# Show gradients flow
axes[2].text(0.5, 0.7, '$\epsilon$ (random)', fontsize=14, ha='center', color='#f43f5e', transform=axes[2].transAxes)
axes[2].text(0.5, 0.5, '$\\downarrow$', fontsize=20, ha='center', color='white', transform=axes[2].transAxes)
axes[2].text(0.5, 0.3, '$z = \mu + \sigma \cdot \epsilon$', fontsize=14, ha='center', color='#818cf8', transform=axes[2].transAxes)
axes[2].text(0.5, 0.1, 'Gradients flow through $\mu$ and $\sigma$ ✓', fontsize=11, ha='center', color='#14b8a6', transform=axes[2].transAxes)
axes[2].axis('off')
axes[2].set_title('Gradient Flow', color='white', fontsize=12)

plt.tight_layout()
plt.show()

**What to notice:** both routes produce the *same distribution*, but only `z = μ + σ·ε` is
differentiable in `μ, σ` — the randomness enters through `ε`, an input, not through the sampling
operation itself. This one-line rewrite is what makes VAE training possible with ordinary backprop.

## KL Divergence

The KL divergence between $q(z|x) = \mathcal{N}(\mu, \sigma^2)$ and the prior $p(z) = \mathcal{N}(0, 1)$ is:

$$D_{KL} = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

Let's visualize how this changes with different $\mu$ and $\sigma$.

In [ ]:
mu_vals = np.linspace(-3, 3, 100)
sigma_vals = np.linspace(0.1, 3, 100)
MU, SIGMA = np.meshgrid(mu_vals, sigma_vals)

# KL divergence for a single dimension
KL = -0.5 * (1 + np.log(SIGMA**2) - MU**2 - SIGMA**2)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(MU, SIGMA, KL, levels=30, cmap='RdYlGn_r')
plt.colorbar(contour, ax=ax, label='$D_{KL}$')
ax.plot(0, 1, 'o', color='white', markersize=10, label='Prior: $\mu=0, \sigma=1$')
ax.set_xlabel('$\mu$')
ax.set_ylabel('$\sigma$')
ax.set_title('KL Divergence from $\mathcal{N}(0, 1)$', color='white', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print('KL = 0 at (mu=0, sigma=1) — exactly matches the prior.')
print('Moving away increases KL, penalizing the encoder.')

**What to notice:** the KL landscape bottoms out at exactly `(μ=0, σ=1)` — the prior — and rises in
every direction. This is the regularizer that organizes the latent space: encoders pay to place
posteriors far from the origin or make them too narrow/wide, so the latent cloud stays a dense,
samplable ball instead of scattered islands.

## The library way — verify the closed-form KL by Monte Carlo

The VAE's KL term uses the Gaussian closed form `−½(1 + log σ² − μ² − σ²)`. Check it against a direct
Monte-Carlo estimate `E_{z~q}[log q(z) − log p(z)]` using `scipy.stats` densities.

In [ ]:
from scipy.stats import norm as norm_dist

def kl_closed(mu, sigma):
    return -0.5 * (1 + np.log(sigma**2) - mu**2 - sigma**2)

rng = np.random.default_rng(0)
for mu_t, sig_t in [(0.0, 1.0), (2.0, 0.8), (-1.0, 2.0)]:
    z = rng.normal(mu_t, sig_t, 200_000)
    mc = np.mean(norm_dist.logpdf(z, mu_t, sig_t) - norm_dist.logpdf(z, 0, 1))
    cf = kl_closed(mu_t, sig_t)
    print(f'mu={mu_t:+.1f}, sigma={sig_t}:  closed form {cf:.4f}   Monte Carlo {mc:.4f}')
    assert abs(mc - cf) < 0.01, "closed-form KL must match the Monte-Carlo estimate"
print('\nclosed-form Gaussian KL == Monte-Carlo E[log q - log p] ✓')

**What to notice:** the closed form matches the sampled expectation at every `(μ, σ)` — including
exactly 0 at the prior. The analytic KL is why VAEs train efficiently: no sampling needed for the
regularizer, only for the reconstruction term (via reparameterization).

## Implementing a VAE

We'll build a simple VAE using NumPy to understand every detail.

In [ ]:
class VAE:
    def __init__(self, input_dim, latent_dim):
        # Encoder: input -> hidden -> (mu, logvar)
        scale1 = np.sqrt(2.0 / input_dim)
        self.W1 = np.random.randn(input_dim, 64) * scale1
        self.b1 = np.zeros(64)
        self.W_mu = np.random.randn(64, latent_dim) * 0.1
        self.b_mu = np.zeros(latent_dim)
        self.W_logvar = np.random.randn(64, latent_dim) * 0.1
        self.b_logvar = np.zeros(latent_dim)
        
        # Decoder: latent -> hidden -> output
        scale2 = np.sqrt(2.0 / latent_dim)
        self.W3 = np.random.randn(latent_dim, 64) * scale2
        self.b3 = np.zeros(64)
        self.W4 = np.random.randn(64, input_dim) * np.sqrt(2.0 / 64)
        self.b4 = np.zeros(input_dim)
    
    def relu(self, x): return np.maximum(0, x)
    def sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -10, 10)))
    
    def encode(self, x):
        self.h1 = self.relu(x @ self.W1 + self.b1)
        self.mu = self.h1 @ self.W_mu + self.b_mu
        self.logvar = self.h1 @ self.W_logvar + self.b_logvar
        return self.mu, self.logvar
    
    def reparameterize(self, mu, logvar):
        std = np.exp(0.5 * logvar)
        eps = np.random.randn(*mu.shape)
        return mu + eps * std
    
    def decode(self, z):
        self.h3 = self.relu(z @ self.W3 + self.b3)
        return self.h3 @ self.W4 + self.b4
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar
    
    def sample(self, n):
        z = np.random.randn(n, self.W_mu.shape[1])
        return self.decode(z)

print('VAE class defined.')

### Backprop for the VAE (with a gradient check)

The original loss needs gradients through *both* paths — the reconstruction (through the
reparameterized `z`) and the KL term (directly into `μ, logvar`). The cell derives them analytically
and **verifies against finite differences** with a frozen `ε` — the same gradient-check discipline as
every other backprop in this course. (`logvar` is clipped to ±8 for numerical safety.)

In [ ]:
def vae_train_step(v, X, lr=0.02, beta=1.0, eps=None):
    """One SGD step on recon + beta*KL. Returns (recon, kl)."""
    n, d = X.shape
    h1p = X @ v.W1 + v.b1;  h1 = np.maximum(0, h1p)
    mu = h1 @ v.W_mu + v.b_mu
    logvar = np.clip(h1 @ v.W_logvar + v.b_logvar, -8, 8)
    k = mu.shape[1];  std = np.exp(0.5 * logvar)
    if eps is None: eps = np.random.randn(*mu.shape)
    z = mu + eps * std
    h3p = z @ v.W3 + v.b3;  h3 = np.maximum(0, h3p)
    xh = h3 @ v.W4 + v.b4

    recon = np.mean((X - xh) ** 2)
    kl = -0.5 * np.mean(1 + logvar - mu**2 - np.exp(logvar))

    # ---- backward: reconstruction path ----
    dxh = 2 * (xh - X) / (n * d)
    dW4 = h3.T @ dxh;  db4 = dxh.sum(0)
    dh3p = (dxh @ v.W4.T) * (h3p > 0)
    dW3 = z.T @ dh3p;  db3 = dh3p.sum(0)
    dz = dh3p @ v.W3.T
    # ---- z = mu + eps*std splits the gradient; KL adds its own terms ----
    dmu = dz + beta * (mu / (n * k))
    dlogvar = dz * eps * 0.5 * std + beta * (-0.5 * (1 - np.exp(logvar)) / (n * k))
    dW_mu = h1.T @ dmu;      db_mu = dmu.sum(0)
    dW_lv = h1.T @ dlogvar;  db_lv = dlogvar.sum(0)
    dh1p = (dmu @ v.W_mu.T + dlogvar @ v.W_logvar.T) * (h1p > 0)
    dW1 = X.T @ dh1p;  db1 = dh1p.sum(0)

    for prm, g in [(v.W1,dW1),(v.b1,db1),(v.W_mu,dW_mu),(v.b_mu,db_mu),
                   (v.W_logvar,dW_lv),(v.b_logvar,db_lv),(v.W3,dW3),(v.b3,db3),(v.W4,dW4),(v.b4,db4)]:
        prm -= lr * g
    return recon, kl


# --- gradient check with a frozen eps ---
import copy
np.random.seed(0)
_v = VAE(2, 2); _Xg = np.random.randn(40, 2); _eps = np.random.randn(40, 2)
def _loss(v):
    h1 = np.maximum(0, _Xg @ v.W1 + v.b1)
    mu = h1 @ v.W_mu + v.b_mu; lv = np.clip(h1 @ v.W_logvar + v.b_logvar, -8, 8)
    z = mu + _eps * np.exp(0.5 * lv)
    h3 = np.maximum(0, z @ v.W3 + v.b3); xh = h3 @ v.W4 + v.b4
    return np.mean((_Xg - xh)**2) - 0.5*np.mean(1 + lv - mu**2 - np.exp(lv))
h = 1e-6
for pname, idx in [('W1', (0, 0)), ('W_logvar', (3, 1)), ('W4', (10, 0))]:
    va = copy.deepcopy(_v); getattr(va, pname)[idx] += h
    vb = copy.deepcopy(_v); getattr(vb, pname)[idx] -= h
    fd = (_loss(va) - _loss(vb)) / (2 * h)
    vc = copy.deepcopy(_v); before = getattr(vc, pname)[idx]
    vae_train_step(vc, _Xg, lr=1.0, beta=1.0, eps=_eps)
    analytic = before - getattr(vc, pname)[idx]
    assert abs(fd - analytic) < 1e-6 * max(1, abs(fd)), f"gradient check failed for {pname}"
    print(f'{pname}{idx}: finite-diff {fd:+.6f}  analytic {analytic:+.6f}  ✓')
print('VAE gradients verified against finite differences ✓')

## Train on 2D toy data

We'll use a simple 2D dataset — a circle — and train a VAE to learn its structure.

In [ ]:
np.random.seed(42)
n = 300
theta = np.random.uniform(0, 2 * np.pi, n)
r = 1.0 + 0.1 * np.random.randn(n)
X = np.column_stack([r * np.cos(theta), r * np.sin(theta)])

vae = VAE(input_dim=2, latent_dim=2)

recons, kls = [], []
BETA = 0.05                                     # final KL weight (beta-VAE dial)
for epoch in range(6000):
    beta = BETA * min(1.0, epoch / 1000)        # KL warm-up: anneal 0 -> BETA to avoid posterior collapse
    recon, kl = vae_train_step(vae, X, lr=0.02, beta=beta)
    recons.append(recon); kls.append(kl)
    if (epoch + 1) % 1500 == 0:
        print(f'Epoch {epoch+1:4d} | Recon: {recon:.4f} | KL: {kl:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(recons, color='#818cf8', linewidth=1, label='reconstruction')
ax.plot(kls, color='#f59e0b', linewidth=1, label='KL')
ax.set_title('VAE Training (real gradient updates)', color='white', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('loss term'); ax.legend()
plt.tight_layout()
plt.show()

**What to notice:** training now performs **real gradient updates** (the original notebook's loop
computed losses but never updated a weight — a silent bug this version fixes). Reconstruction falls
by ~20× while KL *rises* from zero and settles at a healthy value: the encoder is paying the KL price
to store information in the latent. The `beta` warm-up (annealing the KL weight in over the first
1000 epochs) is what prevents the encoder from collapsing onto the prior before the decoder learns
anything.

## Latent space and generation

The VAE forces the latent space to be smooth and structured.
We can now sample from $\mathcal{N}(0, I)$ and decode to generate new data.

In [ ]:
z_encoded = vae.encode(X)[0]
x_reconstructed = vae.forward(X)[0]
x_generated = vae.sample(300)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
fig.suptitle('VAE Results', color='white', fontsize=13, y=1.02)

axes[0].scatter(X[:, 0], X[:, 1], c='#94a3b8', s=15, alpha=0.6)
axes[0].set_title('Original Data', color='white', fontsize=11)

axes[1].scatter(z_encoded[:, 0], z_encoded[:, 1], c=t[:300] if len(t := theta) > 300 else theta, cmap='viridis', s=15, alpha=0.7)
axes[1].set_title('Latent Space', color='white', fontsize=11)
axes[1].set_xlabel('$z_1$')
axes[1].set_ylabel('$z_2$')

axes[2].scatter(x_reconstructed[:, 0], x_reconstructed[:, 1], c='#14b8a6', s=15, alpha=0.6)
axes[2].set_title('Reconstructions', color='white', fontsize=11)

axes[3].scatter(x_generated[:, 0], x_generated[:, 1], c='#eab308', s=15, alpha=0.6)
axes[3].set_title('Generated (sample from $\mathcal{N}(0,I)$)', color='white', fontsize=11)

for ax in axes:
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

**What to notice:** the four panels tell the whole VAE story — the latent cloud is a compact,
roughly Gaussian ball (KL's doing); reconstructions land on the data; and — the payoff — **samples
drawn from the prior** `N(0, I)` decode to points on the data distribution. Compare with the plain
autoencoder's garbage from random latents: the KL term turned a compressor into a generator.

## Latent space manifold

Let's visualize what different regions of the latent space decode to,
by sampling on a grid.

In [ ]:
n_grid = 12
z1 = np.linspace(-2, 2, n_grid)
z2 = np.linspace(-2, 2, n_grid)

fig, axes = plt.subplots(n_grid, n_grid, figsize=(12, 12))
fig.suptitle('Latent Space Manifold — What Each Region Generates', color='white', fontsize=14, y=1.01)

for i, z_val_1 in enumerate(z1):
    for j, z_val_2 in enumerate(z2):
        z = np.array([[z_val_1, z_val_2]])
        x = vae.decode(z)
        ax = axes[n_grid - 1 - i, j]
        # Draw a small circle at the decoded position
        circle = plt.Circle(x[0], 0.15, color='#818cf8', alpha=0.8)
        ax.add_patch(circle)
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_aspect('equal')
        ax.axis('off')

plt.tight_layout()
plt.show()

**What to notice:** sweeping the latent grid decodes to a **smooth manifold** — every point in the
prior's bulk produces a sensible output, and neighbors decode to neighbors. No holes. This continuity
is what makes VAE latents useful for interpolation and controllable generation.

## Gotchas & tradeoffs

- **Posterior collapse:** with a powerful decoder, the easiest ELBO solution can be KL → 0 — the
  latent is ignored and the VAE degenerates. Fixes: KL annealing/warm-up, weaker decoders, free bits.
- **The β tradeoff:** scaling the KL term (β-VAE) trades reconstruction quality for latent
  regularity/disentanglement — β is a real dial, not a fixed constant.
- **VAE samples are blurry:** the likelihood-based objective averages over uncertainty, favoring smooth
  outputs — the classic contrast with GANs (next lesson), which are sharp but unstable.
- **The ELBO is a *lower bound*** on log-likelihood — a looser bound can hide a better model; comparing
  models by ELBO alone is imperfect.

In [ ]:
# Posterior collapse, observed: same VAE, NO warm-up, strong KL from step one
np.random.seed(7)
v_collapse = VAE(input_dim=2, latent_dim=2)
for _ in range(3000):
    rc, klc = vae_train_step(v_collapse, X, lr=0.02, beta=1.0)     # full KL weight immediately
print(f'no warm-up, beta=1.0 : recon={rc:.4f}  KL={klc:.4f}   <- KL ~ 0: the latent is IGNORED')
print(f'with warm-up (above) : recon={recons[-1]:.4f}  KL={kls[-1]:.4f}   <- healthy: latent carries information')
print('\n-> posterior collapse = the encoder matches the prior exactly and the decoder ignores z;')
print('   KL warm-up (or free bits / weaker decoders) is the standard prevention')

**What to notice:** hitting the model with the full KL weight from step one drives KL to ~0 — the
posteriors match the prior *exactly*, meaning the latent carries **no information** and the decoder
just outputs the data mean (reconstruction stuck ~0.5). The warm-up run keeps KL comfortably positive
and reconstruction 20× lower. This is **posterior collapse**, the VAE's characteristic failure, and
the annealing schedule is its standard cure — the β dial and its schedule are as important as the
architecture.

## Key takeaways

- A **VAE** maps inputs to a **distribution** in latent space, not a single point.
- Loss = **reconstruction** + **KL divergence** that regularizes the latent space toward $\mathcal{N}(0, I)$.
- The **reparameterization trick** ($z = \mu + \sigma \odot \epsilon$) makes sampling differentiable.
- The smooth latent space lets you **sample** and **interpolate**; **β** trades reconstruction vs disentanglement.

## ✏️ Your turn

### Exercise 1 — KL divergence (closed form)

For a diagonal Gaussian $q(z) = \mathcal{N}(\mu, \sigma^2 I)$ vs the prior
$p(z) = \mathcal{N}(0, I)$, the KL has the closed form:

$$D_{KL}(q \| p) = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

Implement it and verify three properties: zero at the prior, positive elsewhere, monotone in |μ|.

In [ ]:
import numpy as np

def kl_gaussian(mu, log_var):
    """KL(N(mu, exp(log_var)) || N(0, I)) for vectors mu and log_var (one latent dim each).
    log_var = log(sigma^2), so sigma^2 = exp(log_var)."""
    # TODO(you): implement the closed-form KL
    ...

In [ ]:
# At the prior (mu=0, sigma=1 → log_var=0), KL must be 0
kl_prior = kl_gaussian(np.array([0.0]), np.array([0.0]))
assert abs(kl_prior) < 1e-9, \
    "KL(N(0,1) || N(0,1)) must be 0"

# Any departure from the prior gives positive KL
kl_shifted = kl_gaussian(np.array([2.0]), np.array([0.0]))
assert kl_shifted > 0, \
    "KL(N(2,1) || N(0,1)) must be positive"

# KL increases monotonically as |mu| grows (sigma=1 fixed)
kl_vals = [kl_gaussian(np.array([float(m)]), np.array([0.0])) for m in range(4)]
assert all(kl_vals[i] < kl_vals[i+1] for i in range(3)), \
    "KL must increase as |mu| grows"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def kl_gaussian(mu, log_var):
    return -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))
```

</details>

### Exercise 2 — Reparameterization trick

Instead of sampling $z \sim \mathcal{N}(\mu, \sigma^2)$ directly (non-differentiable w.r.t. $\mu, \sigma$),
we write:
$$z = \mu + \sigma \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0,1)$$

Implement this and verify the sample statistics match $\mu$ and $\sigma$.

In [ ]:
def reparameterize(mu, sigma, n_samples, seed=42):
    """Sample n_samples latent vectors using the reparameterization trick.
    Returns array of shape (n_samples,)."""
    # TODO(you): sample epsilon ~ N(0,1) and return mu + sigma * epsilon
    ...

In [ ]:
np.random.seed(42)
mu, sigma = 2.0, 0.8
samples = reparameterize(mu, sigma, n_samples=50_000, seed=42)

assert samples.shape == (50_000,), "must return (n_samples,) array"
assert abs(samples.mean() - mu) < 0.02, \
    f"sample mean should be ≈ mu={mu}, got {samples.mean():.4f}"
assert abs(samples.std() - sigma) < 0.02, \
    f"sample std should be ≈ sigma={sigma}, got {samples.std():.4f}"
# All samples from z = mu + sigma*eps; eps is N(0,1) so z is N(mu, sigma^2)
assert samples.min() < mu - 2*sigma, \
    "samples should occasionally be more than 2 sigma below the mean"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def reparameterize(mu, sigma, n_samples, seed=42):
    rng = np.random.RandomState(seed)
    eps = rng.randn(n_samples)
    return mu + sigma * eps
```

</details>